# Feature Engineering: Bodenindizes und AEZ × Monat

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Prüfen, ob Bodenindizes (NDTI, BSI) und ein AEZ × Monat-Merkmal die Kulturfehler an den Saisonrändern der hierarchischen RBF-SVM senken (#114).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import CROP_COL, MIN_CV_GAIN, MODEL_DOCS_FIGURES_DIR, MONTH_COL
from awp2.data import load_split
from awp2.evaluation import compare_predictions, error_breakdown
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import make_hierarchical_svm
from awp2.plots import plot_cv_scores, plot_error_rate_by, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothesen

Die Fehleranalyse (#32, #104) zeigt: Rund zwei Drittel der Fehler sind Kulturfehler, gehäuft an
den Saisonrändern (Mai/Juni, Oktober) und in AEZ 9 – vermutlich, weil dort viel offener Boden
das Spektrum prägt. Die mittlere Helligkeit hilft der SVM nicht (#119); sie steckt schon in den
je Band skalierten Spektren.

1. **Bodenindizes** (`use_soil_indices`): NDTI (SWIR 1649/2214 nm) reagiert auf offenen Boden und
   Ernterückstände, BSI trennt Boden von Vegetation. Sie verdichten das Bodensignal in zwei
   Merkmale, die die SVM sonst aus 131 Bändern zusammensetzen muss.
2. **AEZ × Monat** (`use_aez_month`): ein Indikator je beobachteter Kombination aus Region und
   Monat – die Fehler hängen an der Kombination, nicht an AEZ oder Monat allein.

Beide Zweige werden wie die Spektren skaliert. Basis ist die hierarchische RBF-SVM aus #87;
je Variante eine kleine CV-Suche über `gamma` (bisheriger Wert oder `scale`), wie bei den
Vorverarbeitungs-Ablationen in #89. Nur ein CV-Gewinn ≥ `MIN_CV_GAIN` führt zu einem `run()`.

In [ ]:
REFERENCE_PARAMS = {
    "crop_estimator__C": 30.0,
    "crop_estimator__gamma": 0.01,
    "stage_estimator__C": 100.0,
    "stage_estimator__gamma": 0.003,
}
model = make_hierarchical_svm().set_params(**REFERENCE_PARAMS)
gamma_grid = {
    "crop_estimator__gamma": [REFERENCE_PARAMS["crop_estimator__gamma"], "scale"],
    "stage_estimator__gamma": [REFERENCE_PARAMS["stage_estimator__gamma"], "scale"],
}
base = {"use_spectral_standard_scale": True}
variants = {
    "Basis": {},
    "Bodenindizes": {"use_soil_indices": True},
    "AEZ × Monat": {"use_aez_month": True},
    "Bodenindizes + AEZ × Monat": {"use_soil_indices": True, "use_aez_month": True},
}
SLUGS = {
    "Basis": "base",
    "Bodenindizes": "soil",
    "AEZ × Monat": "aez_month",
    "Bodenindizes + AEZ × Monat": "soil_aez_month",
}

## CV-Suche je Variante

In [ ]:
searches = {
    label: tune(
        model,
        TuneConfig(
            name=f"hierarchical_fe_{SLUGS[label]}_search",
            approach="hierarchical",
            description=f"Feature Engineering (#114): {label}; hierarchische RBF-SVM aus #87, "
            "gamma bisher vs. scale je Stufe.",
            preprocessing=PreprocessingConfig(**base, **extra),
            param_grid=gamma_grid,
        ),
    )
    for label, extra in variants.items()
}


def fixed_reference_score(candidates: pd.DataFrame) -> float:
    reference_gamma = {name: REFERENCE_PARAMS[name] for name in gamma_grid}
    return candidates.loc[[p == reference_gamma for p in candidates["params"]], "mean"].item()


base_fixed = fixed_reference_score(searches["Basis"].candidates)
summary = pd.DataFrame(
    {
        label: {
            "CV (bestes gamma)": t.best_score,
            "Gewinn": t.best_score - searches["Basis"].best_score,
            "Gewinn bei festem gamma": fixed_reference_score(t.candidates) - base_fixed,
            "params": t.best_params,
            "run": t.run_id[:8],
        }
        for label, t in searches.items()
    }
).T
summary

In [ ]:
fe_ax = plot_cv_scores(
    {label: t.candidates for label, t in searches.items()},
    reference=searches["Basis"].best_score,
)
fe_ax.set_title("Feature Engineering: bester CV-Score je Variante (gestrichelt: Basis)")
save_doc_figure(fe_ax.figure, "hierarchical_feature_engineering", MODEL_DOCS_FIGURES_DIR)

## Gegenprobe: Kulturfehler nach Monat (Trainingsteil, out-of-fold)

Treffen die neuen Merkmale die Saisonränder? Out-of-fold-Vorhersagen der besten Einstellung je
Variante – nur Trainingsdaten.

In [ ]:
oof = {label: out_of_fold_predictions(t.best_model, t.config) for label, t in searches.items()}
crop_wrong = {label: pred[CROP_COL] != split.y_train[CROP_COL] for label, pred in oof.items()}
month_ax = plot_error_rate_by(crop_wrong, split.X_train[MONTH_COL], xlabel="Monat")
month_ax.set_title("Kulturfehler nach Monat je Variante (Training, out-of-fold)")
save_doc_figure(month_ax.figure, "hierarchical_feature_engineering_month", MODEL_DOCS_FIGURES_DIR)
month_rates = pd.DataFrame(
    {label: wrong.groupby(split.X_train[MONTH_COL]).mean() for label, wrong in crop_wrong.items()}
)
(
    pd.DataFrame(
        {label: error_breakdown(split.y_train, pred).model_dump() for label, pred in oof.items()}
    ).round(4),
    month_rates.round(3),
)

## Einmalige Bewertung

Nur die beste Variante, und nur wenn sie die Basis in der CV um mindestens `MIN_CV_GAIN` schlägt.

In [ ]:
best_label = max(
    (label for label in searches if label != "Basis"), key=lambda label: searches[label].best_score
)
best = searches[best_label]
gain = best.best_score - searches["Basis"].best_score
if gain >= MIN_CV_GAIN:
    result = run(
        best.best_model,
        RunConfig(
            name=f"hierarchical_fe_{SLUGS[best_label]}",
            approach="hierarchical",
            description=f"Beste Feature-Variante ({best_label}) aus #114, einmal bewertet.",
            preprocessing=best.config.preprocessing,
            tuning_run=best.run_id,
        ),
    )
    decision = result.metrics.model_dump()
else:
    result = None
    decision = {"Entscheidung": "Basis bleibt; kein neuer Lauf"}
best_label, round(gain, 4), decision

In [ ]:
if result is not None:
    reference = run(
        model,
        RunConfig(
            name="hierarchical_svm_refit",
            approach="hierarchical",
            preprocessing=PreprocessingConfig(**base),
            track=False,
            log_model=False,
        ),
    )
    validation = (
        {"Basis": reference.metrics.bacc_combined, best_label: result.metrics.bacc_combined},
        pd.DataFrame(
            {
                "Basis": error_breakdown(split.y_val, reference.y_pred).model_dump(),
                best_label: error_breakdown(split.y_val, result.y_pred).model_dump(),
            }
        ).round(4),
        compare_predictions(split.y_val, reference.y_pred, result.y_pred),
    )
else:
    validation = "keine Bewertung"
validation

## Befund

- Keine belastbare Verbesserung. Bodenindizes allein: −0.0003. AEZ × Monat allein: +0.0047 (unter
  `MIN_CV_GAIN`), bei festem gamma +0.0022. Beides: +0.0051, um 0.0001 über der Schwelle – als
  beste von drei Varianten an der Rauschgrenze.
- Hinweis: Mit AEZ × Monat sinken die Kulturfehler im Mai auf dem Trainingsteil von 25,4 % auf
  22,1 % (Kombination 22,8 %); die übrigen Monate kaum.
- Die Kombination wurde einmal bewertet (0.846 vs. 0.844); kein signifikanter Unterschied
  (McNemar p = 0.63), Kulturfehler gleich.
- Ob der Effekt aus der Kombination stammt oder nur aus mehr Gewicht für den Kontext im
  RBF-Abstand, zeigt der Versuch nicht.